# 🎾 Feature Engineering для Tennis Match Prediction

## Мета:
Створити фічі для ML моделі на основі оброблених даних з `Prepare_ETL.ipynb`

## Основні групи фіч:
1. **Seed Features** - binary indicators, seed difference, seed tiers
2. **Rolling Statistics** - середні показники з останніх N матчів
3. **Head-to-Head (H2H)** - історія зустрічей між гравцями
4. **Surface/Tournament** - encoding покриття та рівня турніру
5. **Rank-based Features** - різниця рангів, співвідношення
6. **P1 vs P2 Format** - конвертація з winner/loser в player1/player2

In [60]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("✅ Бібліотеки завантажено")

✅ Бібліотеки завантажено


## 📥 Крок 1: Завантаження оброблених даних

In [61]:
# Завантажуємо обробленні дані з ETL
processed_path = Path('../data/processed')

df_train = pd.read_csv(processed_path / 'train_2012_2024.csv')
df_test = pd.read_csv(processed_path / 'test_2025.csv')

print("📊 ДАНІ ЗАВАНТАЖЕНО")
print("=" * 80)
print(f"Train (2012-2024): {len(df_train):,} матчів, {len(df_train.columns)} колонок")
print(f"Test (2025):       {len(df_test):,} матчів, {len(df_test.columns)} колонок")
print("=" * 80)

# Об'єднуємо для створення фіч (потім знову розділимо)
df_all = pd.concat([df_train, df_test], ignore_index=True)
print(f"\nОб'єднано: {len(df_all):,} матчів")

# Конвертуємо дату в datetime
df_all['tourney_date'] = pd.to_datetime(df_all['tourney_date'], format='%Y%m%d', errors='coerce')

# Сортуємо за датою (важливо для rolling features)
df_all = df_all.sort_values('tourney_date').reset_index(drop=True)

print(f"✅ Дані відсортовані за датою: {df_all['tourney_date'].min()} → {df_all['tourney_date'].max()}")

📊 ДАНІ ЗАВАНТАЖЕНО
Train (2012-2024): 36,624 матчів, 50 колонок
Test (2025):       2,911 матчів, 50 колонок

Об'єднано: 39,535 матчів
✅ Дані відсортовані за датою: 2012-01-01 00:00:00 → 2025-11-10 00:00:00


## 🎯 Крок 2: Seed Features

Seed - один з найсильніших сигналів після H2H!

In [62]:
print("🎯 СТВОРЕННЯ SEED FEATURES")
print("=" * 80)

# 1. Binary indicators: чи гравець посіяний?
df_all['is_winner_seeded'] = df_all['winner_seed'].notna().astype(int)
df_all['is_loser_seeded'] = df_all['loser_seed'].notna().astype(int)

seeded_w = df_all['is_winner_seeded'].sum()
seeded_l = df_all['is_loser_seeded'].sum()
print(f"✓ Binary indicators створено:")
print(f"  Winner посіяні: {seeded_w:,} ({seeded_w/len(df_all)*100:.1f}%)")
print(f"  Loser посіяні:  {seeded_l:,} ({seeded_l/len(df_all)*100:.1f}%)")

# 2. Seed difference (якщо обидва посіяні)
df_all['seed_diff'] = df_all['winner_seed'] - df_all['loser_seed']
# NaN замінюємо на 0 (хоча б один не посіяний)
df_all['seed_diff'] = df_all['seed_diff'].fillna(0)

both_seeded = (df_all['is_winner_seeded'] == 1) & (df_all['is_loser_seeded'] == 1)
print(f"\n✓ Seed difference створено:")
print(f"  Матчів де обидва посіяні: {both_seeded.sum():,} ({both_seeded.sum()/len(df_all)*100:.1f}%)")
print(f"  Середня різниця seed: {df_all.loc[both_seeded, 'seed_diff'].mean():.2f}")

# 3. Seed tiers (для посіяних гравців)
def seed_tier(seed):
    if pd.isna(seed):
        return 'Not_Seeded'
    elif seed <= 4:
        return 'Top4'
    elif seed <= 8:
        return 'Top8'
    elif seed <= 16:
        return 'Top16'
    else:
        return 'Top32+'

df_all['winner_seed_tier'] = df_all['winner_seed'].apply(seed_tier)
df_all['loser_seed_tier'] = df_all['loser_seed'].apply(seed_tier)

print(f"\n✓ Seed tiers створено:")
print(f"  Winner seed tiers:")
print(df_all['winner_seed_tier'].value_counts().sort_index())

print("=" * 80)

🎯 СТВОРЕННЯ SEED FEATURES
✓ Binary indicators створено:
  Winner посіяні: 16,696 (42.2%)
  Loser посіяні:  9,510 (24.1%)

✓ Seed difference створено:
  Матчів де обидва посіяні: 4,092 (10.4%)
  Середня різниця seed: -3.43

✓ Seed tiers створено:
  Winner seed tiers:
winner_seed_tier
Not_Seeded    22839
Top16          2719
Top32+         1899
Top4           7004
Top8           5074
Name: count, dtype: int64


## 📈 Крок 3: Rolling Statistics

**Найсильніший сигнал!** Середні показники з останніх N матчів для кожного гравця.

In [65]:
print("📈 СТВОРЕННЯ ROLLING STATISTICS (БЕЗ DATA LEAKAGE!)")
print("=" * 80)

# 🚨 КРИТИЧНО: Обчислюємо rolling stats для КОЖНОГО ГРАВЦЯ окремо!
# НЕ для winner/loser (бо це leakage), а для player_id незалежно від результату

# Додаємо row_id для унікальності (бо гравець може грати 2+ матчі в один день)
df_all = df_all.reset_index(drop=True)
df_all['row_id'] = df_all.index

# Статистичні колонки
stats_cols = [
    'ace', 'df', 'svpt', '1stIn', '1stWon', '2ndWon',
    'SvGms', 'bpSaved', 'bpFaced'
]

# Параметри rolling window
WINDOW_SIZE = 10  # останні 10 матчів
MIN_PERIODS = 3   # мінімум 3 матчі для обчислення

print(f"Параметри: window={WINDOW_SIZE}, min_periods={MIN_PERIODS}\n")
print("🔄 КРОК 1: Створюємо long-format датасет (кожен гравець = окремий рядок)")
print("-" * 80)

# Створюємо два датафрейми: один для winner, один для loser
winner_df = df_all[[
    'tourney_date', 'row_id', 'winner_id', 'winner_name',
    'w_ace', 'w_df', 'w_svpt', 'w_1stIn', 'w_1stWon', 'w_2ndWon',
    'w_SvGms', 'w_bpSaved', 'w_bpFaced'
]].copy()

loser_df = df_all[[
    'tourney_date', 'row_id', 'loser_id', 'loser_name',
    'l_ace', 'l_df', 'l_svpt', 'l_1stIn', 'l_1stWon', 'l_2ndWon',
    'l_SvGms', 'l_bpSaved', 'l_bpFaced'
]].copy()

# Перейменовуємо колонки в єдиний формат
winner_df.columns = ['date', 'row_id', 'player_id', 'player_name'] + stats_cols
loser_df.columns = ['date', 'row_id', 'player_id', 'player_name'] + stats_cols

# Об'єднуємо в один датафрейм
player_stats = pd.concat([winner_df, loser_df], ignore_index=True)
player_stats = player_stats.sort_values(['player_id', 'date', 'row_id']).reset_index(drop=True)

print(f"✅ Long-format створено: {len(player_stats):,} рядків (гравець-матч)")
print(f"   Унікальних гравців: {player_stats['player_id'].nunique():,}")

print("\n🔄 КРОК 2: Обчислюємо rolling statistics для кожного гравця")
print("-" * 80)

# Обчислюємо rolling для кожної статистики
for stat in stats_cols:
    col_name = f"{stat}_roll{WINDOW_SIZE}"
    
    # Групуємо за player_id і обчислюємо rolling
    player_stats[col_name] = player_stats.groupby('player_id')[stat].transform(
        lambda x: x.shift(1).rolling(window=WINDOW_SIZE, min_periods=MIN_PERIODS).mean()
    )
    
    non_null = player_stats[col_name].notna().sum()
    print(f"  ✓ {col_name:<20} заповнено: {non_null:>6,} ({non_null/len(player_stats)*100:>5.1f}%)")

print("\n🔄 КРОК 3: Маппимо rolling stats назад на winner/loser")
print("-" * 80)

# Створюємо унікальний lookup ключ (player_id + row_id)
player_stats['lookup_key'] = (
    player_stats['player_id'].astype(str) + '_' + 
    player_stats['row_id'].astype(str)
)

# Додаємо lookup key до основного датафрейму
df_all['winner_lookup'] = (
    df_all['winner_id'].astype(str) + '_' + 
    df_all['row_id'].astype(str)
)
df_all['loser_lookup'] = (
    df_all['loser_id'].astype(str) + '_' + 
    df_all['row_id'].astype(str)
)

# Створюємо словник для швидкого lookup
roll_cols = [f"{stat}_roll{WINDOW_SIZE}" for stat in stats_cols]
lookup_dict = player_stats.set_index('lookup_key')[roll_cols].to_dict('index')

# Заповнюємо rolling stats для winner (префікс w_)
for stat in stats_cols:
    roll_col = f"{stat}_roll{WINDOW_SIZE}"
    w_col = f"w_{roll_col}"
    
    df_all[w_col] = df_all['winner_lookup'].map(
        lambda x: lookup_dict.get(x, {}).get(roll_col, np.nan)
    )

# Заповнюємо rolling stats для loser (префікс l_)
for stat in stats_cols:
    roll_col = f"{stat}_roll{WINDOW_SIZE}"
    l_col = f"l_{roll_col}"
    
    df_all[l_col] = df_all['loser_lookup'].map(
        lambda x: lookup_dict.get(x, {}).get(roll_col, np.nan)
    )

# Видаляємо тимчасові колонки
df_all = df_all.drop(['winner_lookup', 'loser_lookup', 'row_id'], axis=1)

print("✅ Маппінг завершено!")

# Статистика покриття
w_roll_filled = df_all['w_ace_roll10'].notna().sum()
l_roll_filled = df_all['l_ace_roll10'].notna().sum()
print(f"\n📊 Покриття rolling features:")
print(f"   Winner rolling stats: {w_roll_filled:,} ({w_roll_filled/len(df_all)*100:.1f}%)")
print(f"   Loser rolling stats:  {l_roll_filled:,} ({l_roll_filled/len(df_all)*100:.1f}%)")

print("\n" + "=" * 80)
print("✅ Rolling statistics створено БЕЗ DATA LEAKAGE!")
print("   ✅ Кожен гравець обробляється незалежно від результату")
print("   ✅ Використовуємо shift(1) - не включаємо поточний матч")
print("   ✅ Групування по player_id, не по winner/loser")
print("=" * 80)

📈 СТВОРЕННЯ ROLLING STATISTICS (БЕЗ DATA LEAKAGE!)
Параметри: window=10, min_periods=3

🔄 КРОК 1: Створюємо long-format датасет (кожен гравець = окремий рядок)
--------------------------------------------------------------------------------
✅ Long-format створено: 79,070 рядків (гравець-матч)
   Унікальних гравців: 1,554

🔄 КРОК 2: Обчислюємо rolling statistics для кожного гравця
--------------------------------------------------------------------------------
  ✓ ace_roll10           заповнено: 74,013 ( 93.6%)
  ✓ df_roll10            заповнено: 74,013 ( 93.6%)
  ✓ svpt_roll10          заповнено: 74,013 ( 93.6%)
  ✓ df_roll10            заповнено: 74,013 ( 93.6%)
  ✓ svpt_roll10          заповнено: 74,013 ( 93.6%)
  ✓ 1stIn_roll10         заповнено: 74,013 ( 93.6%)
  ✓ 1stWon_roll10        заповнено: 74,013 ( 93.6%)
  ✓ 1stIn_roll10         заповнено: 74,013 ( 93.6%)
  ✓ 1stWon_roll10        заповнено: 74,013 ( 93.6%)
  ✓ 2ndWon_roll10        заповнено: 74,013 ( 93.6%)
  ✓ SvGms_roll10

## 🤝 Крок 4: Head-to-Head (H2H) Features

Історія зустрічей між двома гравцями.

In [66]:
print("🤝 СТВОРЕННЯ HEAD-TO-HEAD FEATURES")
print("=" * 80)

# Створюємо унікальний ключ для пари гравців (сортований)
def create_h2h_key(row):
    players = sorted([row['winner_name'], row['loser_name']])
    return f"{players[0]}_vs_{players[1]}"

df_all['h2h_key'] = df_all.apply(create_h2h_key, axis=1)

# Ініціалізуємо колонки
df_all['h2h_winner_wins'] = 0
df_all['h2h_loser_wins'] = 0
df_all['h2h_total_matches'] = 0
df_all['h2h_winner_win_rate'] = 0.5  # default

# Словник для збереження H2H історії
h2h_history = {}

print("Обробка матчів для H2H...")
print("-" * 80)

# Ітеруємо по матчах в хронологічному порядку
for idx, row in df_all.iterrows():
    h2h_key = row['h2h_key']
    winner = row['winner_name']
    loser = row['loser_name']
    
    # Отримуємо поточну H2H статистику
    if h2h_key not in h2h_history:
        h2h_history[h2h_key] = {winner: 0, loser: 0}
    
    # Зберігаємо статистику ДО цього матчу
    winner_wins = h2h_history[h2h_key].get(winner, 0)
    loser_wins = h2h_history[h2h_key].get(loser, 0)
    total = winner_wins + loser_wins
    
    df_all.at[idx, 'h2h_winner_wins'] = winner_wins
    df_all.at[idx, 'h2h_loser_wins'] = loser_wins
    df_all.at[idx, 'h2h_total_matches'] = total
    
    if total > 0:
        df_all.at[idx, 'h2h_winner_win_rate'] = winner_wins / total
    
    # Оновлюємо історію ПІСЛЯ цього матчу
    if winner not in h2h_history[h2h_key]:
        h2h_history[h2h_key][winner] = 0
    h2h_history[h2h_key][winner] += 1
    
    # Прогрес (кожні 5000 матчів)
    if (idx + 1) % 5000 == 0:
        print(f"  Оброблено: {idx + 1:,} / {len(df_all):,} матчів ({(idx+1)/len(df_all)*100:.1f}%)")

print("-" * 80)

# Статистика H2H
has_history = df_all['h2h_total_matches'] > 0
print(f"\n✅ H2H Features створено:")
print(f"  Унікальних пар гравців: {len(h2h_history):,}")
print(f"  Матчів з H2H історією: {has_history.sum():,} ({has_history.sum()/len(df_all)*100:.1f}%)")
print(f"  Середня кількість попередніх зустрічей: {df_all.loc[has_history, 'h2h_total_matches'].mean():.2f}")

print("=" * 80)

🤝 СТВОРЕННЯ HEAD-TO-HEAD FEATURES
Обробка матчів для H2H...
--------------------------------------------------------------------------------
  Оброблено: 5,000 / 39,535 матчів (12.6%)
  Оброблено: 10,000 / 39,535 матчів (25.3%)
  Оброблено: 5,000 / 39,535 матчів (12.6%)
  Оброблено: 10,000 / 39,535 матчів (25.3%)
  Оброблено: 15,000 / 39,535 матчів (37.9%)
  Оброблено: 20,000 / 39,535 матчів (50.6%)
  Оброблено: 15,000 / 39,535 матчів (37.9%)
  Оброблено: 20,000 / 39,535 матчів (50.6%)
  Оброблено: 25,000 / 39,535 матчів (63.2%)
  Оброблено: 30,000 / 39,535 матчів (75.9%)
  Оброблено: 25,000 / 39,535 матчів (63.2%)
  Оброблено: 30,000 / 39,535 матчів (75.9%)
  Оброблено: 35,000 / 39,535 матчів (88.5%)
--------------------------------------------------------------------------------

✅ H2H Features створено:
  Унікальних пар гравців: 22,961
  Матчів з H2H історією: 16,574 (41.9%)
  Середня кількість попередніх зустрічей: 2.40
  Оброблено: 35,000 / 39,535 матчів (88.5%)
------------------

## 🎾 Крок 5: Surface & Tournament Features

In [67]:
print("🎾 СТВОРЕННЯ SURFACE & TOURNAMENT FEATURES")
print("=" * 80)

# 1. Tournament level encoding
tourney_level_map = {
    'G': 4,  # Grand Slam (найважливіший)
    'M': 3,  # Masters 1000
    'A': 2,  # ATP 500/250
    'D': 1,  # Davis Cup / інші
    'F': 1   # Tour Finals
}

df_all['tourney_level_encoded'] = df_all['tourney_level'].map(tourney_level_map).fillna(1)

print("✓ Tournament level encoding:")
print(df_all.groupby('tourney_level')['tourney_level_encoded'].first().sort_values(ascending=False))

# 2. Surface encoding (one-hot не потрібно для tree-based моделей)
surface_map = {
    'Hard': 3,
    'Clay': 2,
    'Grass': 1,
    'Carpet': 0
}

df_all['surface_encoded'] = df_all['surface'].map(surface_map).fillna(3)  # default Hard

print("\n✓ Surface encoding:")
print(df_all.groupby('surface')['surface_encoded'].first().sort_values(ascending=False))

# 3. Indoor/Outdoor (заповнюємо NaN як 0 і конвертуємо в int)
if 'indoor' in df_all.columns:
    # Спочатку заповнюємо NaN
    df_all['indoor'] = df_all['indoor'].fillna(0)
    # Потім конвертуємо в numeric (якщо є текст)
    df_all['indoor'] = pd.to_numeric(df_all['indoor'], errors='coerce').fillna(0).astype(int)
    
    indoor_count = df_all['indoor'].sum()
    print(f"\n✓ Indoor/Outdoor:")
    print(f"  Indoor матчів: {indoor_count:,} ({indoor_count/len(df_all)*100:.1f}%)")
    print(f"  Outdoor матчів: {len(df_all) - indoor_count:,} ({(len(df_all) - indoor_count)/len(df_all)*100:.1f}%)")
else:
    print(f"\n⚠️  Indoor колонка відсутня в датасеті")

print("=" * 80)

🎾 СТВОРЕННЯ SURFACE & TOURNAMENT FEATURES
✓ Tournament level encoding:
tourney_level
G      4.0
M      3.0
A      2.0
250    1.0
500    1.0
D      1.0
F      1.0
O      1.0
Name: tourney_level_encoded, dtype: float64

✓ Surface encoding:
surface
Hard      3.0
Clay      2.0
Grass     1.0
Carpet    0.0
Name: surface_encoded, dtype: float64

✓ Indoor/Outdoor:
  Indoor матчів: 0 (0.0%)
  Outdoor матчів: 39,535 (100.0%)


## 📊 Крок 6: Базові ранги (без Data Leakage)

⚠️ **ВАЖЛИВО**: Не створюємо rank_diff/rank_ratio тут, бо це було б **data leakage**!  
Rank features створюються ПІСЛЯ конвертації в P1/P2 формат.

In [68]:
print("📊 СТВОРЕННЯ RANK-BASED FEATURES")
print("=" * 80)

# ⚠️ ВАЖЛИВО: НЕ використовуємо winner/loser для rank features!
# Це було б data leakage, бо ми вже знаємо результат.
# Замість цього створюємо базові фічі, які потім будуть використані в P1/P2 форматі.

# Зберігаємо базові ранги без порівняння winner/loser
# (порівняння буде в P1/P2 форматі після конвертації)

print("✓ Базові rank features збережено (без leakage)")
print("  winner_rank, loser_rank - будуть конвертовані в p1_rank, p2_rank")
print("  winner_rank_points, loser_rank_points - будуть конвертовані в p1_rank_points, p2_rank_points")
print("\n⚠️  Rank difference, rank ratio та інші порівняння")
print("   будуть створені ПІСЛЯ конвертації в P1/P2 формат!")

print("=" * 80)

📊 СТВОРЕННЯ RANK-BASED FEATURES
✓ Базові rank features збережено (без leakage)
  winner_rank, loser_rank - будуть конвертовані в p1_rank, p2_rank
  winner_rank_points, loser_rank_points - будуть конвертовані в p1_rank_points, p2_rank_points

⚠️  Rank difference, rank ratio та інші порівняння
   будуть створені ПІСЛЯ конвертації в P1/P2 формат!


## 🔄 Крок 7: Конвертація в P1 vs P2 формат

Зараз дані в форматі winner/loser. Для моделі потрібен формат P1/P2 з таргетом (P1_won).

In [69]:
print("🔄 КОНВЕРТАЦІЯ В P1 vs P2 ФОРМАТ")
print("=" * 80)

# Список колонок для конвертації (виключаємо H2H колонки які вже створені)
winner_cols = [col for col in df_all.columns if col.startswith('winner_') and not col.startswith('winner_seed_tier')]
loser_cols = [col for col in df_all.columns if col.startswith('loser_') and not col.startswith('loser_seed_tier')]
w_stats = [col for col in df_all.columns if col.startswith('w_') and '_roll' not in col]
l_stats = [col for col in df_all.columns if col.startswith('l_') and '_roll' not in col]

print(f"Колонок для конвертації:")
print(f"  winner_*: {len(winner_cols)}")
print(f"  loser_*: {len(loser_cols)}")
print(f"  w_*: {len(w_stats)}")
print(f"  l_*: {len(l_stats)}")
print("\nСтворюємо дублікати матчів (P1=Winner, P2=Loser) та (P1=Loser, P2=Winner)...\n")
print("-" * 80)

# Варіант 1: P1 = Winner, P2 = Loser, target = 1
df_p1_wins = df_all.copy()

# Перейменовуємо колонки
rename_dict = {}

# Основні колонки гравців
for col in winner_cols:
    new_name = col.replace('winner_', 'p1_')
    rename_dict[col] = new_name

for col in loser_cols:
    new_name = col.replace('loser_', 'p2_')
    rename_dict[col] = new_name

# Статистика матчу (оригінальні колонки без _roll)
for col in w_stats:
    new_name = col.replace('w_', 'p1_')
    rename_dict[col] = new_name

for col in l_stats:
    new_name = col.replace('l_', 'p2_')
    rename_dict[col] = new_name

# Rolling statistics
for col in df_all.columns:
    if '_roll' in col:
        if col.startswith('w_'):
            rename_dict[col] = col.replace('w_', 'p1_')
        elif col.startswith('l_'):
            rename_dict[col] = col.replace('l_', 'p2_')

# Seed tiers
if 'winner_seed_tier' in df_all.columns:
    rename_dict['winner_seed_tier'] = 'p1_seed_tier'
if 'loser_seed_tier' in df_all.columns:
    rename_dict['loser_seed_tier'] = 'p2_seed_tier'

# H2H колонки
if 'h2h_winner_wins' in df_all.columns:
    rename_dict['h2h_winner_wins'] = 'h2h_p1_wins'
if 'h2h_loser_wins' in df_all.columns:
    rename_dict['h2h_loser_wins'] = 'h2h_p2_wins'
if 'h2h_winner_win_rate' in df_all.columns:
    rename_dict['h2h_winner_win_rate'] = 'h2h_p1_win_rate'

# Seeded indicators
if 'is_winner_seeded' in df_all.columns:
    rename_dict['is_winner_seeded'] = 'is_p1_seeded'
if 'is_loser_seeded' in df_all.columns:
    rename_dict['is_loser_seeded'] = 'is_p2_seeded'

df_p1_wins = df_p1_wins.rename(columns=rename_dict)
df_p1_wins['p1_won'] = 1  # P1 виграв

print(f"✓ Варіант 1 створено: {len(df_p1_wins):,} матчів (P1=Winner)")

# Варіант 2: P1 = Loser, P2 = Winner, target = 0
df_p2_wins = df_all.copy()

# Міняємо місцями
rename_dict_swap = {}

# Основні колонки гравців (міняємо місцями!)
for col in winner_cols:
    new_name = col.replace('winner_', 'p2_')
    rename_dict_swap[col] = new_name

for col in loser_cols:
    new_name = col.replace('loser_', 'p1_')
    rename_dict_swap[col] = new_name

# Статистика матчу (міняємо місцями!)
for col in w_stats:
    new_name = col.replace('w_', 'p2_')
    rename_dict_swap[col] = new_name

for col in l_stats:
    new_name = col.replace('l_', 'p1_')
    rename_dict_swap[col] = new_name

# Rolling statistics (міняємо місцями!)
for col in df_all.columns:
    if '_roll' in col:
        if col.startswith('w_'):
            rename_dict_swap[col] = col.replace('w_', 'p2_')
        elif col.startswith('l_'):
            rename_dict_swap[col] = col.replace('l_', 'p1_')

# Seed tiers (міняємо місцями!)
if 'winner_seed_tier' in df_all.columns:
    rename_dict_swap['winner_seed_tier'] = 'p2_seed_tier'
if 'loser_seed_tier' in df_all.columns:
    rename_dict_swap['loser_seed_tier'] = 'p1_seed_tier'

# H2H колонки (міняємо місцями!)
if 'h2h_winner_wins' in df_all.columns:
    rename_dict_swap['h2h_winner_wins'] = 'h2h_p2_wins'
if 'h2h_loser_wins' in df_all.columns:
    rename_dict_swap['h2h_loser_wins'] = 'h2h_p1_wins'
if 'h2h_winner_win_rate' in df_all.columns:
    rename_dict_swap['h2h_winner_win_rate'] = 'h2h_p2_win_rate'

# Seeded indicators (міняємо місцями!)
if 'is_winner_seeded' in df_all.columns:
    rename_dict_swap['is_winner_seeded'] = 'is_p2_seeded'
if 'is_loser_seeded' in df_all.columns:
    rename_dict_swap['is_loser_seeded'] = 'is_p1_seeded'

df_p2_wins = df_p2_wins.rename(columns=rename_dict_swap)
df_p2_wins['p1_won'] = 0  # P1 програв

# Також міняємо місцями h2h_p1_win_rate для варіанту 2
if 'h2h_p2_win_rate' in df_p2_wins.columns:
    df_p2_wins['h2h_p1_win_rate'] = 1 - df_p2_wins['h2h_p2_win_rate']
    df_p2_wins = df_p2_wins.drop(columns=['h2h_p2_win_rate'])

print(f"✓ Варіант 2 створено: {len(df_p2_wins):,} матчів (P1=Loser)")

# Об'єднуємо обидва варіанти
df_final = pd.concat([df_p1_wins, df_p2_wins], ignore_index=True)

print("\n" + "-" * 80)
print(f"✅ ФІНАЛЬНИЙ ДАТАСЕТ: {len(df_final):,} рядків (х2 від оригіналу)")

# ============================================================================
# ТЕПЕР СТВОРЮЄМО RANK FEATURES БЕЗ DATA LEAKAGE (в P1/P2 форматі)
# ============================================================================
print("\n🔧 СТВОРЕННЯ RANK FEATURES (БЕЗ LEAKAGE)")
print("-" * 80)

# 1. Rank difference (P1 - P2, без знання результату)
df_final['rank_diff'] = df_final['p1_rank'] - df_final['p2_rank']

# 2. Rank ratio
df_final['rank_ratio'] = df_final['p1_rank'] / df_final['p2_rank']

# 3. Rank points difference
df_final['rank_points_diff'] = df_final['p1_rank_points'] - df_final['p2_rank_points']

# 4. Is P1 favorite? (нижчий rank = фаворит)
df_final['is_p1_favorite'] = (df_final['p1_rank'] < df_final['p2_rank']).astype(int)

print(f"✓ rank_diff створено (P1 - P2)")
print(f"✓ rank_ratio створено (P1 / P2)")
print(f"✓ rank_points_diff створено")
print(f"✓ is_p1_favorite створено (1 якщо P1 має кращий rank)")

# Статистика
favorites = df_final['is_p1_favorite'].sum()
favorites_won = df_final[df_final['is_p1_favorite'] == 1]['p1_won'].sum()
print(f"\n📊 Статистика:")
print(f"  P1 фаворит: {favorites:,} матчів ({favorites/len(df_final)*100:.1f}%)")
print(f"  Фаворити виграли: {favorites_won:,} ({favorites_won/favorites*100:.1f}%)")

print("-" * 80)

print(f"\nРозподіл таргета:")
print(df_final['p1_won'].value_counts().sort_index())
print(f"\nБаланс: {df_final['p1_won'].mean()*100:.1f}% P1 виграли")

print("=" * 80)

🔄 КОНВЕРТАЦІЯ В P1 vs P2 ФОРМАТ
Колонок для конвертації:
  winner_*: 10
  loser_*: 10
  w_*: 9
  l_*: 9

Створюємо дублікати матчів (P1=Winner, P2=Loser) та (P1=Loser, P2=Winner)...

--------------------------------------------------------------------------------
✓ Варіант 1 створено: 39,535 матчів (P1=Winner)
✓ Варіант 2 створено: 39,535 матчів (P1=Loser)

--------------------------------------------------------------------------------
✅ ФІНАЛЬНИЙ ДАТАСЕТ: 79,070 рядків (х2 від оригіналу)

🔧 СТВОРЕННЯ RANK FEATURES (БЕЗ LEAKAGE)
--------------------------------------------------------------------------------
✓ rank_diff створено (P1 - P2)
✓ rank_ratio створено (P1 / P2)
✓ rank_points_diff створено
✓ is_p1_favorite створено (1 якщо P1 має кращий rank)

📊 Статистика:
  P1 фаворит: 39,412 матчів (49.8%)
  Фаворити виграли: 25,821 (65.5%)
--------------------------------------------------------------------------------

Розподіл таргета:
p1_won
0    39535
1    39535
Name: count, dtype: int

## 🧹 Крок 8: Фінальна підготовка та збереження

In [76]:
print("🧹 ФІНАЛЬНА ПІДГОТОВКА")
print("=" * 80)

# 🚨 КРИТИЧНО: Видаляємо оригінальні статистичні колонки (DATA LEAKAGE!)
# Ці колонки - це результат поточного матчу, тому вони не можуть використовуватись

leakage_cols = [
    # Статистика поточного матчу (P1)
    'p1_ace', 'p1_df', 'p1_svpt', 'p1_1stIn', 'p1_1stWon', 'p1_2ndWon',
    'p1_SvGms', 'p1_bpSaved', 'p1_bpFaced',
    # Статистика поточного матчу (P2)
    'p2_ace', 'p2_df', 'p2_svpt', 'p2_1stIn', 'p2_1stWon', 'p2_2ndWon',
    'p2_SvGms', 'p2_bpSaved', 'p2_bpFaced',
    # Результати матчу
    'score', 'best_of', 'round', 'minutes',
]

# Видаляємо leakage колонки
leakage_existing = [col for col in leakage_cols if col in df_final.columns]
print(f"🚨 Видаляємо {len(leakage_existing)} leakage колонок:")
for col in leakage_existing:
    print(f"  ❌ {col}")

df_final = df_final.drop(columns=leakage_existing)

print(f"\n✅ Залишилось колонок: {len(df_final.columns)}")

# Розділяємо на train/test за data_year
df_train_final = df_final[df_final['data_year'] < 2025].copy()
df_test_final = df_final[df_final['data_year'] == 2025].copy()

print(f"\nTrain (2012-2024): {len(df_train_final):,} рядків")
print(f"Test (2025):       {len(df_test_final):,} рядків")
print(f"Співвідношення:    {len(df_train_final) / len(df_test_final):.1f}:1")

# Перевірка критичних фіч
critical_features = [
    'p1_rank', 'p2_rank',
    'is_p1_seeded', 'is_p2_seeded',
    'rank_diff', 'rank_ratio',
    'h2h_total_matches', 'h2h_p1_win_rate',
    'tourney_level_encoded', 'surface_encoded',
    'p1_won'
]

# Перевіряємо які фічі існують
available_critical = [f for f in critical_features if f in df_final.columns]

print(f"\nКритичні фічі в датасеті: {len(available_critical)} / {len(critical_features)}")

# Список всіх створених фіч (БЕЗ leakage!)
feature_cols = [
    col for col in df_final.columns 
    if col.startswith(('p1_', 'p2_', 'rank_', 'seed_', 'h2h_', 'tourney_', 'surface_', 'is_'))
    and col != 'p1_won'
]

print(f"\nВсього створено фіч (БЕЗ LEAKAGE): {len(feature_cols)}")
print(f"\nПриклади фіч:")
for feat in sorted(feature_cols)[:15]:
    print(f"  • {feat}")

print("\n" + "=" * 80)

🧹 ФІНАЛЬНА ПІДГОТОВКА
🚨 Видаляємо 0 leakage колонок:

✅ Залишилось колонок: 63

Train (2012-2024): 73,248 рядків
Test (2025):       5,822 рядків
Співвідношення:    12.6:1

Критичні фічі в датасеті: 11 / 11

Всього створено фіч (БЕЗ LEAKAGE): 58

Приклади фіч:
  • h2h_key
  • h2h_p1_win_rate
  • h2h_p1_wins
  • h2h_p2_wins
  • h2h_total_matches
  • is_p1_favorite
  • is_p1_seeded
  • is_p2_seeded
  • p1_1stIn_roll10
  • p1_1stWon_roll10
  • p1_2ndWon_roll10
  • p1_SvGms_roll10
  • p1_ace_roll10
  • p1_age
  • p1_bpFaced_roll10



In [77]:
# Збереження результатів
features_path = Path('../data/processed')
features_path.mkdir(exist_ok=True, parents=True)

train_features_path = features_path / 'train_features.csv'
test_features_path = features_path / 'test_features.csv'

df_train_final.to_csv(train_features_path, index=False)
df_test_final.to_csv(test_features_path, index=False)

print("💾 ЗБЕРЕЖЕННЯ ФІНАЛНИХ ДАНИХ")
print("=" * 80)
print(f"✓ Train збережено: {train_features_path}")
print(f"  Розмір: {train_features_path.stat().st_size / (1024**2):.2f} MB")
print(f"  Рядків: {len(df_train_final):,}")
print(f"  Колонок: {len(df_train_final.columns)}")
print()
print(f"✓ Test збережено: {test_features_path}")
print(f"  Розмір: {test_features_path.stat().st_size / (1024**2):.2f} MB")
print(f"  Рядків: {len(df_test_final):,}")
print(f"  Колонок: {len(df_test_final.columns)}")
print("=" * 80)
print("\n✅ Feature Engineering завершено!")
print("\n🎯 Готово до тренування моделі!")

💾 ЗБЕРЕЖЕННЯ ФІНАЛНИХ ДАНИХ
✓ Train збережено: ../data/processed/train_features.csv
  Розмір: 26.97 MB
  Рядків: 73,248
  Колонок: 63

✓ Test збережено: ../data/processed/test_features.csv
  Розмір: 2.18 MB
  Рядків: 5,822
  Колонок: 63

✅ Feature Engineering завершено!

🎯 Готово до тренування моделі!


## 📝 Підсумок створених фіч

### 1. **Seed Features**
- `is_p1_seeded`, `is_p2_seeded` - binary indicators
- `seed_diff` - різниця посівів (P1 - P2)
- `p1_seed_tier`, `p2_seed_tier` - категорії (Top4, Top8, Top16, Top32+)

### 2. **Rolling Statistics** (window=10, min_periods=3)
- `p1_ace_roll10`, `p2_ace_roll10`
- `p1_df_roll10`, `p2_df_roll10`
- `p1_1stIn_roll10`, `p2_1stIn_roll10`
- і т.д. для всіх статистичних показників
- ✅ Використовується `shift(1)` - БЕЗ data leakage!

### 3. **Head-to-Head**
- `h2h_p1_wins`, `h2h_p2_wins`
- `h2h_total_matches`
- `h2h_p1_win_rate`
- ✅ Рахується ДО поточного матчу - БЕЗ data leakage!

### 4. **Surface & Tournament**
- `tourney_level_encoded` (1-4)
- `surface_encoded` (0-3)
- `indoor` (0/1)

### 5. **Rank-based** (створені ПІСЛЯ P1/P2 конвертації)
- `rank_diff` - різниця рангів (P1 - P2)
- `rank_ratio` - співвідношення (P1 / P2)
- `rank_points_diff` - різниця очок
- `is_p1_favorite` - чи P1 має кращий rank (1/0)
- ✅ БЕЗ використання winner/loser - БЕЗ data leakage!

### 6. **Target**
- `p1_won` - бінарний таргет (0/1)

---

## ✅ **Захист від Data Leakage:**

1. ✅ Rolling features з `shift(1)` - не використовуємо поточний матч
2. ✅ H2H рахується ДО матчу, а не після
3. ✅ Rank features створені в P1/P2 форматі (не winner/loser)
4. ✅ Видалено `minutes`, `score` - це результати матчу
5. ✅ Статистика матчу (`w_ace`, `l_df`) використовується тільки для rolling features

**Датасет збалансований 50/50 завдяки дублюванню матчів!**